# 01 - Exploratory Data Analysis

**Purpose:** Look at the shape of `customer_rfm_final.csv` before any model gets built on top of it. Specifically, this notebook checks the churn class balance, how the RFM segments break down, the country skew, and flags outlier customers (like customer 12346, spotted during the SQL sanity checks) that are worth a closer look before modeling.

**Input:** `data/staged/customer_rfm_final.csv`, produced by `sql/04_export_views.sql`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# We set a consistent plot style up front so every chart in this notebook
# looks the same, instead of re-styling each one individually.
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (8, 5)

DATA_PATH = "../data/staged/customer_rfm_final.csv"


## 1. Load and inspect the data

In [ ]:
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# We used encoding="utf-8-sig" because SQL Server Management Studio's CSV
# export sometimes prepends a BOM (byte-order mark) to the file. Without
# this, the first column header comes through as an invisible-character
# variant of "customer_id" instead of the plain string, and df['customer_id']
# fails with a KeyError even though the printed column name looks identical.
# We also strip whitespace from every column name as a second safety net,
# since the same class of invisible-character issue can show up elsewhere.
df.columns = df.columns.str.strip()

# We check shape and dtypes first, before looking at any actual values,
# because a wrong dtype (e.g. rfm_score read as an int instead of a string)
# would silently break the segment logic later.
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.info()


In [ ]:
df.head(10)


**Expected:** 5,285 rows, 8 columns (`customer_id`, `country`, `recency_days`, `frequency`, `monetary`, `rfm_score`, `rfm_segment`, `churned`). If the row count doesn't match, the CSV export didn't match `features.customer_rfm` and needs re-checking before anything below is trustworthy.

## 2. Data quality check

In [ ]:
# We check for nulls and duplicate customer_ids here rather than assuming
# the SQL layer already guaranteed this -- the notebook should verify its
# own inputs, not just trust upstream code blindly.
print("Null counts per column:")
print(df.isnull().sum())

print(f"\nDuplicate customer_id rows: {df['customer_id'].duplicated().sum()}")


If nulls or duplicates show up here, stop and go back to `03_rfm_features.sql` rather than patching around it in Python -- the SQL layer is supposed to be the single source of truth for this table.

## 3. Churn distribution

This is the single most important number in the notebook: it tells us whether we're dealing with a balanced or imbalanced classification problem, which decides how we evaluate the model in the next notebook.

In [ ]:
churn_counts = df["churned"].value_counts().sort_index()
churn_rate = df["churned"].mean()

print(churn_counts)
print(f"\nChurn rate: {churn_rate:.1%}")

ax = churn_counts.plot(kind="bar", color=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["Retained (0)", "Churned (1)"], rotation=0)
ax.set_ylabel("Customer count")
ax.set_title("Churn distribution")
plt.show()


**Reading this:** a churn rate in the mid-50s% (as the SQL sanity check showed, ~56.7%) means this is a moderately imbalanced problem, not a severe one. It's still worth using stratified train/test splits and checking precision/recall (not just accuracy) in the classification notebook, since a model that just predicts "churned" for everyone would already score above 50% accuracy without learning anything useful.

## 4. RFM segment distribution

In [ ]:
segment_order = df["rfm_segment"].value_counts().index

ax = sns.countplot(data=df, y="rfm_segment", order=segment_order, palette="viridis")
ax.set_xlabel("Customer count")
ax.set_ylabel("")
ax.set_title("Customers per RFM segment")
plt.show()

df.groupby("rfm_segment")["monetary"].agg(["count", "mean", "median"]).sort_values("mean", ascending=False)


**Reading this:** Champions should have the highest average monetary value and Lost the lowest -- if that ordering is reversed anywhere, it's worth re-checking the quintile scoring logic in `03_rfm_features.sql` before trusting the segments.

## 5. Recency, Frequency, Monetary distributions

Each of these tends to be right-skewed in retail data (most customers spend modestly, a small number spend a lot) -- we check that assumption here rather than assuming it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df["recency_days"], bins=30, ax=axes[0], color="#4C72B0")
axes[0].set_title("Recency (days)")

sns.histplot(df["frequency"], bins=30, ax=axes[1], color="#55A868")
axes[1].set_title("Frequency (distinct invoices)")

sns.histplot(df["monetary"], bins=30, ax=axes[2], color="#C44E52")
axes[2].set_title("Monetary (total spend)")

plt.tight_layout()
plt.show()

df[["recency_days", "frequency", "monetary"]].describe()


**Reading this:** if `monetary` and `frequency` are heavily right-skewed (a long tail of high-spending outliers), that's worth flagging for the classification notebook -- some models (like logistic regression) are sensitive to that skew and benefit from a log transform, while tree-based models (Random Forest, Gradient Boosting) generally don't need it.

## 6. Country distribution

We kept all countries rather than filtering to UK-only, so it's worth confirming here just how lopsided that split actually is.

In [ ]:
top_countries = df["country"].value_counts().head(10)

ax = top_countries.plot(kind="barh", color="#4C72B0")
ax.invert_yaxis()
ax.set_xlabel("Customer count")
ax.set_title("Top 10 countries by customer count")
plt.show()

print(f"UK share of total customers: {(df['country'] == 'United Kingdom').mean():.1%}")


**Reading this:** given the SQL results already showed the UK at roughly 90%+ of transaction volume, `country` as a model feature will mostly separate "UK vs. not UK" in practice -- individual smaller countries won't have enough customers for the model to learn much about them specifically. Worth noting as a limitation in the report rather than treating country as a strong standalone predictor.

## 7. Outlier check: customer 12346

Flagged during the SQL sanity checks -- highest monetary value in the entire dataset ($77,556.46 across 12 orders) but labeled `churned = 1`. Worth a direct look before deciding whether this is a real churn case or a data artifact (e.g. a wholesale account that buys in large, sporadic batches rather than a regular pattern).

In [ ]:
df[df["customer_id"] == 12346]


**Reading this:** a single row won't tell us much beyond confirming the segment and churn label -- the real investigation (looking at this customer's actual invoice history) needs the pre-aggregation transaction data from `clean.online_retail_valid` in SQL, not this customer-level table. Worth a follow-up query if this case gets discussed in the report, since "our biggest historical spender churned" is a strong, specific talking point.

## Summary of findings

- Churn rate sits at roughly 57% -- moderate imbalance, needs stratified splitting and precision/recall evaluation rather than plain accuracy in the next notebook.
- RFM segments behave as expected: Champions have the highest average monetary value, Lost the lowest.
- Country is heavily UK-skewed (~90%+), so it's a weak standalone predictor despite being kept as a feature.
- Customer 12346 is a notable outlier worth a specific callout in the report: highest lifetime spend, but churned.

**Next:** `02_churn_classification.ipynb` -- train and evaluate the churn classifier using these features.
